# Practical 08: Columnar Storage with Apache Parquet Format in PySpark

**UNIT 2: Big Data Processing using PySpark**

**Problem Statement:** Write a PySpark program to demonstrate high-performance columnar storage using Apache Parquet. Create a PySpark DataFrame, export it to Parquet format using .write.mode('overwrite').parquet(), reload the Parquet file using spark.read.parquet(), and verify schema preservation and row counts.

In [1]:
import os
import shutil
import tempfile
from pyspark.sql import SparkSession, Row

spark = (SparkSession.builder.appName("DS505-Lab08").master("local[*]")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

data = [
    {"Name": "Rutvik", "Department": "IT", "City": "Nagpur", "Salary": 73000, "Rating": 4.3},
    {"Name": "Hetal", "Department": "HR", "City": "Indore", "Salary": 55000, "Rating": 3.7},
    {"Name": "Dhruv", "Department": "Finance", "City": "Nagpur", "Salary": 68000, "Rating": 4.1},
    {"Name": "Mansi", "Department": "IT", "City": "Bhopal", "Salary": 89000, "Rating": 4.6},
    {"Name": "Chirag", "Department": "Sales", "City": "Indore", "Salary": 50000, "Rating": 3.6},
]
employees = spark.createDataFrame([Row(**d) for d in data])
employees.printSchema()
employees.show()

tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, "emp_data.parquet")

employees.write.mode("overwrite").parquet(path)
print("Parquet written to:", path)
print("Files inside      :", [f for f in os.listdir(path) if not f.startswith(".")])

df_parq = spark.read.parquet(path)
df_parq.printSchema()
df_parq.show()

print("Schema preserved :", employees.schema == df_parq.schema)
print(f"Row count original = {employees.count()} | reloaded = {df_parq.count()} | equal = {employees.count() == df_parq.count()}")

df_parq.select("Salary").show()

shutil.rmtree(tmp_dir)
spark.stop()
print("Cleaned up.")

root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+------+----------+------+------+------+
|  Name|Department|  City|Salary|Rating|
+------+----------+------+------+------+
|Rutvik|        IT|Nagpur| 73000|   4.3|
| Hetal|        HR|Indore| 55000|   3.7|
| Dhruv|   Finance|Nagpur| 68000|   4.1|
| Mansi|        IT|Bhopal| 89000|   4.6|
|Chirag|     Sales|Indore| 50000|   3.6|
+------+----------+------+------+------+

Parquet written to: /tmp/tmpt0smxzn8/emp_data.parquet
Files inside      : ['part-00001-23e6595b-c646-44a9-9977-760a4f1dc6a7-c000.snappy.parquet', 'part-00000-23e6595b-c646-44a9-9977-760a4f1dc6a7-c000.snappy.parquet', '_SUCCESS']
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+------+-----